# Длительная динамика выбранных узлов

Сохраняется ли динамика выбранных гауссовых импульсов до $T=400$?
Пять узлов выбраны по наибольшему изменению поля на $T=100$:
по возможности по одному с единственным пиком и с несколькими пиками
или переменным их числом; при отсутствии одной группы — два из другой.
Нестационарных узлов в режиме $F=2$, $\Delta=3.5$ не было.

Сравниваются исходный запуск до $T=100$ и повтор той же начальной задачи
до $T=400$. Фиксированы $\beta_2=-0.1$, $L=2\pi$, $N=512$, $\Delta t=0.0005$.

In [ ]:
import csv
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from lle_runs.project import get_experiment_dir
from lle_runs.runs import get_run
from lle_solver.simulations.gaussian import GaussianConfig
from lle_solver.simulations.storage import load_evolution
from lle_analysis.fields import time_mask, relative_field_l2_error, spatial_contrast
from lle_analysis.peaks import peak_indices, peak_position
from lle_solver.grid import PeriodicGrid
from lle_analysis.criteria import DEFAULT_STATIONARY_TOLERANCE, is_stationary
from lle_viz.plots import plot_intensity, plot_profiles, plot_separation
from lle_viz.style import configure_plots, save_figure


In [ ]:
EXPERIMENT_DIR = get_experiment_dir(4)
RESULTS_DIR = EXPERIMENT_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

In [ ]:
configure_plots()

In [ ]:
with (EXPERIMENT_DIR / "long_time_pairs.csv").open(encoding="utf-8") as source:
    selected = list(csv.DictReader(source))

## Измерения временного поведения

Сравниваем окна одинаковой длительности 20: $[80,100]$ и $[380,400]$. На каждом измеряем относительное изменение комплексного поля относительно последнего снимка окна, контраст и количество пиков. Стационарность проверяется с порогом $10^{-3}$.

Число пиков измеряется с выраженностью 25% размаха интенсивности. Оно не определяет стационарность; при малом контрасте детектор может выделять шум. Как в эксперименте 2, рассматриваем изменение поля, движение пиков и изменение локальной формы отдельно, без классификации режимов.


In [ ]:
observation_duration = 20.
traces = {}
long_evolutions = {}

In [ ]:
observations = []
for pair in selected:
    pair_results = []
    for run_key, observation_time in (("base_run_id", 100.), ("long_run_id", 400.)):
        row = dict(run_id=pair[run_key])
        run = get_run(row["run_id"])
        config = run.load_config(GaussianConfig)
        evolution = load_evolution(run.path)
        if run_key == "long_run_id":
            long_evolutions[run.id] = evolution
        mask = time_mask(evolution.times, 0., observation_time)
        window = evolution[mask]
        assert np.isclose(window.times[-1], observation_time, rtol=0, atol=1e-10)
        tail = window[time_mask(window.times, observation_time - observation_duration, observation_time)]
        assert tail.times.size >= 5
        assert np.isclose(tail.times[0], observation_time-observation_duration, rtol=0, atol=1e-10)
        intensity = tail.intensity
        change = float(relative_field_l2_error(tail, tail[-1]).max())
        contrast = float(spatial_contrast(intensity).max())
        stationary = is_stationary(tail)
        counts = sorted({len(peak_indices(profile)) for profile in intensity})
        traces[(pair["long_run_id"], observation_time)] = dict(
            times=tail.times, field_error=relative_field_l2_error(tail, tail[-1]),
            contrast=spatial_contrast(intensity),
            peak_counts=np.array([len(peak_indices(profile)) for profile in intensity]))

        pair_results.append(dict(stationary=stationary, window_start=float(tail.times[0]), field_change=change,
                                 contrast_max=contrast, peak_counts=counts))
    before, after = pair_results
    observations.append(dict(F=config.equation.pump_amplitude, delta=config.equation.detuning,
                             A=config.initial.amplitude, sigma=config.initial.width,
                             stationary_T100=before["stationary"], window_start_T100=before["window_start"],
                             stationary_T400=after["stationary"], window_start_T400=after["window_start"],
                             contrast_T100=before["contrast_max"], contrast_T400=after["contrast_max"],
                             field_change_T100=before["field_change"], field_change_T400=after["field_change"],
                             peak_counts_T100=before["peak_counts"], peak_counts_T400=after["peak_counts"],
                             base_run_id=pair["base_run_id"], run_id=pair["long_run_id"]))

In [ ]:
for pair, observation in zip(selected, observations):
    figure, axes = plt.subplots(3, 1, figsize=(8, 8), sharex=True, layout="constrained")
    for observation_time in (100., 400.):
        trace = traces[(pair["long_run_id"], observation_time)]
        elapsed = trace["times"] - trace["times"][0]
        label = f'T={observation_time:g}'
        axes[0].plot(elapsed, trace["field_error"], label=label)
        axes[1].plot(elapsed, trace["contrast"], label=label)
        axes[2].step(elapsed, trace["peak_counts"], where="mid", label=label)
    axes[0].set(ylabel="Изменение поля", title=f'F={observation["F"]:g}, delta={observation["delta"]:g}, A={observation["A"]:g}, sigma={observation["sigma"]:.3g}')
    axes[0].set_yscale("symlog", linthresh=1e-5)
    axes[0].set_ylim(bottom=0)
    axes[2].yaxis.set_major_locator(MaxNLocator(integer=True))
    axes[2].set_ylim(min(line.get_ydata().min() for line in axes[2].lines) - .5,
                     max(line.get_ydata().max() for line in axes[2].lines) + .5)
    axes[0].axhline(DEFAULT_STATIONARY_TOLERANCE, color="black", linestyle="--")
    axes[1].set(ylabel="Контраст")
    axes[2].set(xlabel="Время от начала окна", ylabel="Число пиков")
    for ax in axes:
        ax.legend()
    save_figure(RESULTS_DIR, "time_metrics_" + pair["long_run_id"], figure)
    plt.show()

## Движение пиков

Показываем координаты всех найденных пиков. Как в эксперименте 2, расстояние вычисляем только для снимков с ровно двумя пиками; остальные значения пропускаем. Облако точек не является автоматическим отслеживанием идентичности пиков.


In [ ]:
peak_coordinates = {}
separations = {}
positions = {}
for run_id, evolution in long_evolutions.items():
    coordinates = []
    for field in evolution:
        intensity = field.intensity
        coordinates.append([peak_position(intensity, int(i)) for i in peak_indices(intensity)])
    peak_coordinates[run_id] = coordinates
    positions[run_id] = np.full((len(coordinates), 2), np.nan)
    for index, peaks in enumerate(coordinates):
        if len(peaks) == 2:
            positions[run_id][index] = peaks
    grid = PeriodicGrid.from_theta(evolution.theta)
    separations[run_id] = np.array([grid.distance(first, second)
                                   for first, second in positions[run_id]])

for row in observations:
    run_id = row["run_id"]
    evolution = long_evolutions[run_id]
    figure, axes = plt.subplots(2, 1, sharex=True, figsize=(8, 6), layout="constrained")
    coordinates = peak_coordinates[run_id]
    axes[0].scatter(np.repeat(evolution.times, [len(peaks) for peaks in coordinates]),
                    [position for peaks in coordinates for position in peaks], s=2)
    axes[0].set(ylabel=r"Координаты пиков $\theta$",
                title=f'F={row["F"]:g}, Δ={row["delta"]:g}, A={row["A"]:g}, σ={row["sigma"]:.3g}')
    plot_separation(axes[1], evolution.times, separations[run_id])
    axes[1].set(xlabel="Время", ylabel="Расстояние между двумя пиками")
    if not np.isfinite(separations[run_id]).any():
        axes[1].text(.5, .5, "Нет снимков с ровно двумя пиками", transform=axes[1].transAxes, ha="center")
    save_figure(RESULTS_DIR, "peak_motion_" + run_id, figure)
    plt.show()


## Форма в движущихся координатах

Как в эксперименте 2, сравниваем комплексные профили в окне $[-0.4,0.4]$ вокруг двух пиков на интервале $[380,400]$. Расчёт выполняется только если два пика присутствуют на каждом снимке. Пики сопоставляются по порядку координат; пересечение периодической границы или обмен порядком требует отдельного отслеживания. Для остальных запусков локальная форма здесь не оценивается.


In [ ]:
offsets = np.linspace(-.4, .4, 129)
shape_traces = {}
for run_id, evolution in long_evolutions.items():
    indices = np.flatnonzero(time_mask(evolution.times, 400.-observation_duration, 400.))
    centers = positions[run_id][indices]
    if np.isnan(centers).any():
        continue
    snapshots = [[evolution[index].relative_to(center).field_at(offsets) for center in pair]
                 for index, pair in zip(indices, centers)]
    reference = snapshots[-1]
    shape_traces[run_id] = np.array([[relative_field_l2_error(profile, final)
                                    for profile, final in zip(profiles, reference, strict=True)]
                                   for profiles in snapshots])
    row = next(row for row in observations if row["run_id"] == run_id)
    figure, axes = plt.subplots(2, 1, sharex=True, figsize=(8, 6), layout="constrained")
    plot_separation(axes[0], evolution.times[indices], separations[run_id][indices])
    axes[0].set(ylabel="Расстояние между пиками",
                title=f'F={row["F"]:g}, Δ={row["delta"]:g}, A={row["A"]:g}, σ={row["sigma"]:.3g}')
    for index in range(2):
        axes[1].plot(evolution.times[indices], shape_traces[run_id][:, index], label=f"Пик {index+1}")
    axes[1].axhline(DEFAULT_STATIONARY_TOLERANCE, color="black", linestyle="--")
    axes[1].set(xlabel="Время", ylabel="Относительное изменение формы", ylim=(0, None))
    axes[1].legend()
    save_figure(RESULTS_DIR, "moving_shape_" + run_id, figure)
    plt.show()
if not shape_traces:
    print("На последнем окне нет запусков с двумя пиками на каждом снимке.")


## Эволюции и профили

### Случай 1: $F=1.8$, $\Delta=3$, $A=3.2$, $\sigma\approx0.3293$

In [ ]:
run = get_run(selected[0]["long_run_id"])
evolution = load_evolution(run.path)
intensity = evolution.intensity

In [ ]:
figure, ax = plt.subplots(layout="constrained")
image = plot_intensity(ax, intensity)
ax.set(xlabel=r"$\theta$", ylabel="$t$")
figure.colorbar(image, ax=ax, label=r"$|\psi|^2$")
save_figure(RESULTS_DIR, "long_evolution_F_lower_2026-11-03_06-58-35_758488", figure)
plt.show()

In [ ]:
profile_times = (0., 320., 400.)
profiles = [evolution.at_time(time).intensity for time in profile_times]

In [ ]:
figure, ax = plt.subplots(layout="constrained")
lines = plot_profiles(ax, profiles)
for line, time in zip(lines, profile_times):
    line.set_label(f"$T={time:g}$")
ax.set(xlabel=r"$\theta$", ylabel=r"$|\psi|^2$")
ax.legend()
save_figure(RESULTS_DIR, "long_profiles_F_lower_2026-11-03_06-58-35_758488", figure)
plt.show()

### Случай 2: $F=2.2$, $\Delta=3$, $A=0.4$, $\sigma\approx0.1063$

In [ ]:
run = get_run(selected[1]["long_run_id"])
evolution = load_evolution(run.path)
intensity = evolution.intensity

In [ ]:
figure, ax = plt.subplots(layout="constrained")
image = plot_intensity(ax, intensity)
ax.set(xlabel=r"$\theta$", ylabel="$t$")
figure.colorbar(image, ax=ax, label=r"$|\psi|^2$")
save_figure(RESULTS_DIR, "long_evolution_F_higher_2026-11-03_07-03-00_803863", figure)
plt.show()

In [ ]:
profile_times = (0., 320., 400.)
profiles = [evolution.at_time(time).intensity for time in profile_times]

In [ ]:
figure, ax = plt.subplots(layout="constrained")
lines = plot_profiles(ax, profiles)
for line, time in zip(lines, profile_times):
    line.set_label(f"$T={time:g}$")
ax.set(xlabel=r"$\theta$", ylabel=r"$|\psi|^2$")
ax.legend()
save_figure(RESULTS_DIR, "long_profiles_F_higher_2026-11-03_07-03-00_803863", figure)
plt.show()

### Случай 3: $F=2.2$, $\Delta=3$, $A=3.2$, $\sigma\approx0.0500$

In [ ]:
run = get_run(selected[2]["long_run_id"])
evolution = load_evolution(run.path)
intensity = evolution.intensity

In [ ]:
figure, ax = plt.subplots(layout="constrained")
image = plot_intensity(ax, intensity)
ax.set(xlabel=r"$\theta$", ylabel="$t$")
figure.colorbar(image, ax=ax, label=r"$|\psi|^2$")
save_figure(RESULTS_DIR, "long_evolution_F_higher_2026-11-03_07-03-00_844861", figure)
plt.show()

In [ ]:
profile_times = (0., 320., 400.)
profiles = [evolution.at_time(time).intensity for time in profile_times]

In [ ]:
figure, ax = plt.subplots(layout="constrained")
lines = plot_profiles(ax, profiles)
for line, time in zip(lines, profile_times):
    line.set_label(f"$T={time:g}$")
ax.set(xlabel=r"$\theta$", ylabel=r"$|\psi|^2$")
ax.legend()
save_figure(RESULTS_DIR, "long_profiles_F_higher_2026-11-03_07-03-00_844861", figure)
plt.show()

### Случай 4: $F=2$, $\Delta=2.5$, $A=0.8$, $\sigma\approx0.3293$

In [ ]:
run = get_run(selected[3]["long_run_id"])
evolution = load_evolution(run.path)
intensity = evolution.intensity

In [ ]:
figure, ax = plt.subplots(layout="constrained")
image = plot_intensity(ax, intensity)
ax.set(xlabel=r"$\theta$", ylabel="$t$")
figure.colorbar(image, ax=ax, label=r"$|\psi|^2$")
save_figure(RESULTS_DIR, "long_evolution_delta_lower_2026-11-03_07-08-07_826902", figure)
plt.show()

In [ ]:
profile_times = (0., 320., 400.)
profiles = [evolution.at_time(time).intensity for time in profile_times]

In [ ]:
figure, ax = plt.subplots(layout="constrained")
lines = plot_profiles(ax, profiles)
for line, time in zip(lines, profile_times):
    line.set_label(f"$T={time:g}$")
ax.set(xlabel=r"$\theta$", ylabel=r"$|\psi|^2$")
ax.legend()
save_figure(RESULTS_DIR, "long_profiles_delta_lower_2026-11-03_07-08-07_826902", figure)
plt.show()

### Случай 5: $F=2$, $\Delta=2.5$, $A=0.8$, $\sigma\approx0.0729$

In [ ]:
run = get_run(selected[4]["long_run_id"])
evolution = load_evolution(run.path)
intensity = evolution.intensity

In [ ]:
figure, ax = plt.subplots(layout="constrained")
image = plot_intensity(ax, intensity)
ax.set(xlabel=r"$\theta$", ylabel="$t$")
figure.colorbar(image, ax=ax, label=r"$|\psi|^2$")
save_figure(RESULTS_DIR, "long_evolution_delta_lower_2026-11-03_07-08-07_873922", figure)
plt.show()

In [ ]:
profile_times = (0., 320., 400.)
profiles = [evolution.at_time(time).intensity for time in profile_times]

In [ ]:
figure, ax = plt.subplots(layout="constrained")
lines = plot_profiles(ax, profiles)
for line, time in zip(lines, profile_times):
    line.set_label(f"$T={time:g}$")
ax.set(xlabel=r"$\theta$", ylabel=r"$|\psi|^2$")
ax.legend()
save_figure(RESULTS_DIR, "long_profiles_delta_lower_2026-11-03_07-08-07_873922", figure)
plt.show()